In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
src = str(ROOT / "src")
if src not in sys.path:
    sys.path.insert(0, src)
from bs_accel.data import generate_dataset, split_dataset
from bs_accel.model import build_mlp
from bs_accel.evaluate import load_trained_model
from bs_accel.train import save_checkpoint


# Implied volatility from Black-Scholes

Samples use the package's fixed seed. Ten percent is held out for the final test;
validation uses ten percent of the remaining rows. All searches use validation loss.
Old outputs and scores were cleared because this split changes the experiment.


In [ ]:
X, y = generate_dataset("iv")


In [ ]:
from math import exp

import optuna
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler

X_train, X_test, y_train, y_test = split_dataset(X, y, "iv")
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.1, random_state=0
)


# Architecture search on validation data

The original architecture used 434 units, 6 hidden layers and batch size 1090.
This run selects its architecture on validation data.


In [ ]:
def objective(trial):
    n_layers = trial.suggest_int("n_layers", 2, 6)
    n_units = trial.suggest_int("n_units", 200, 600)
    batch_size = trial.suggest_int("batch_size", 1024, 2048)
    trial_model = build_mlp(X_train.shape[1], n_units, n_layers)
    history = trial_model.fit(
        X_train, y_train, epochs=1200, batch_size=batch_size, verbose=1,
        validation_data=(X_val, y_val),
        callbacks=[LearningRateScheduler(
            lambda epoch, lr: lr if epoch < 1000 else lr * exp(-0.1)
        )],
    )
    return history.history["val_loss"][-1]

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)
best_params = study.best_params
print("Best architecture:", best_params, "validation MSE:", study.best_value)


# Train phase one and save a complete checkpoint


In [ ]:
batch_size = best_params["batch_size"]
model = build_mlp(X_train.shape[1], best_params["n_units"], best_params["n_layers"])
model.fit(
    X_train, y_train, epochs=1200, batch_size=batch_size, verbose=1,
    validation_data=(X_val, y_val),
)
phase1_checkpoint = ROOT / "model" / "notebook-iv" / "phase1.keras"
save_checkpoint(model, phase1_checkpoint)
print("Phase-one validation MSE:", model.evaluate(X_val, y_val, verbose=0))


# Learning-rate search from the same phase-one checkpoint


In [ ]:
def fit_phase2(model, exp_rate):
    model.fit(
        X_train, y_train, epochs=2000, batch_size=batch_size, verbose=1,
        validation_data=(X_val, y_val),
        callbacks=[
            LearningRateScheduler(lambda _epoch, lr: lr * exp(exp_rate)),
            EarlyStopping(monitor="val_loss", patience=100, restore_best_weights=True),
        ],
    )


def objective(trial):
    trial_model = load_trained_model(phase1_checkpoint)
    fit_phase2(trial_model, trial.suggest_float("exp_rate", -0.1, -0.001))
    return trial_model.evaluate(X_val, y_val, verbose=0)

learning_study = optuna.create_study(direction="minimize")
learning_study.optimize(objective, n_trials=50)
print("Best learning rate:", learning_study.best_params)
print("Validation MSE:", learning_study.best_value)


# Train the selected model and evaluate the test split once


In [ ]:
model = load_trained_model(phase1_checkpoint)
fit_phase2(model, learning_study.best_params["exp_rate"])
print("Final test MSE:", model.evaluate(X_test, y_test, verbose=0))


In [ ]:
save_checkpoint(model, ROOT / "model" / "notebook-iv" / "model.keras")
